# Afinado consciente de la cuantización (QAT) en Colab

Afina con cuantización simulada el punto de partida elegido en la sección 8: el baseline de SH17,
que es la rama que el apartado 5.2 compara con PTQ, o un modelo podado de la rama de poda y QAT.

El resultado son pesos FP32, no un modelo INT8. El INT8 lo materializa después cada runtime con el
mismo conjunto de calibración (128 imágenes, hash `a27c6f11fb1865fd`) y por el mismo camino que la
rama PTQ, para que la diferencia entre ramas se deba a la técnica.

Se ejecuta de arriba abajo en una sesión limpia. Solo hay que editar `PUNTO_DE_PARTIDA`, al
principio de la sección 8. Cada punto de partida lleva sus cifras medidas en local, que las
secciones 8, 9 y 11 contrastan.

| # | Celda | Qué hace |
|---|---|---|
| 1 | Drive | Monta Drive: dataset, pesos de partida y salida de la campaña |
| 2 | Repositorio | Clona el repositorio |
| 3 | Entorno | Crea un venv con Python 3.12, instala dentro y comprueba los paquetes |
| 4 | Dataset | Descomprime `resized_640` y comprueba que Ultralytics lo acepta |
| 5 | Referencia | Restaura el COCO y las particiones con las que se midió el baseline |
| 6 | Entorno | `00_setup_check.py` |
| 7 | Máquina | Registra la GPU asignada, junto a los checkpoints en Drive |
| 8 | Pesos | Elige el punto de partida y comprueba SHA-256, parámetros y FP32 |
| 9 | Preparación | Prepara QAT sin afinar y compara con lo medido en local |
| 10 | Presupuesto | Coste estimado con la cifra medida del baseline |
| 11 | Campaña | El afinado |
| 12 | Resultados | Copia registro, volcado crudo y checkpoints |

Diferencias con el cuaderno 01 (poda):

- La campaña cabe en una sesión: el mínimo son unos 20 min.
- No hay reanudación. Si la sesión se corta, relanzar falla con el motivo en lugar de empezar de
  cero encima.
- El presupuesto de la sección 10 es un mínimo. El modelo preparado lleva 64 observadores de peso y
  121 de activación (72 y 137 en un podado), y su coste real queda en el `training_cost` de la
  campaña.

El `Trainer` pasa explícitamente el optimizador a Ultralytics; si no, su `auto` descartaría la tasa
configurada y un QAT de 10 épocas correría AdamW a 4,76e-4 en lugar de SGD a 1e-4. La sección 9 lo
comprueba antes de usar la GPU, y el `Trainer` lo vuelve a comprobar en cada entrenamiento.

El proyecto no se ejecuta con el Python de Colab: la imagen trae Python 3.13 y los pines de
`requirements.txt` no tienen rueda para esa versión. La sección 3 crea un venv con CPython 3.12 y
todo lo que usa el proyecto (`tfm_edge`, `torch`, `ultralytics` y los `scripts/`) corre ahí. En el
kernel solo quedan Drive, los secretos y la biblioteca estándar.

## 1. Montar Drive

Drive guarda el dataset y los checkpoints, que tienen que sobrevivir al cierre de la sesión, así que
el entrenamiento escribe directamente ahí.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/tfm-edge")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

DATASET_TAR = DRIVE_BASE / "sh17_resized640.tar.gz"
RUNS_DIR = DRIVE_BASE / "runs"
RUNS_DIR.mkdir(exist_ok=True)

assert DRIVE_BASE.is_dir(), f"No se ha montado Drive en {DRIVE_BASE}"
print(f"Drive montado. Base del proyecto: {DRIVE_BASE}")
print(f"  dataset esperado : {DATASET_TAR}  ({'presente' if DATASET_TAR.is_file() else 'FALTA'})")
print(f"  checkpoints en   : {RUNS_DIR}")

if not DATASET_TAR.is_file():
    disponibles = sorted(p.name for p in DRIVE_BASE.glob("*"))
    raise FileNotFoundError(
        f"Falta '{DATASET_TAR}'.\n"
        f"Lo que hay en {DRIVE_BASE}: {disponibles}\n"
        "Súbelo antes de continuar (ver la portada de este cuaderno)."
    )

Salida esperada: la ruta de Drive y el dataset como `presente`. Si aparece `FALTA`, la celda se
detiene y muestra el contenido del directorio.

## 2. Clonar el repositorio

El repositorio es público y se clona sin credenciales. Si existe el secreto `GITHUB_TOKEN`, se usa
en la URL de clonado y los mensajes de error de git se censuran antes de mostrarlos.

In [ ]:
import shutil
import subprocess

from google.colab import userdata

REPO_DIR = Path("/content/tfm-edge-object-detection")

# El repositorio es público, así que el secreto 'GITHUB_TOKEN' es opcional.
try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    GITHUB_TOKEN = ""


def _censurar(texto: str) -> str:
    """Quita el token de un texto antes de mostrarlo."""
    return texto.replace(GITHUB_TOKEN, "[TOKEN]") if GITHUB_TOKEN else texto


if REPO_DIR.is_dir():
    print(f"El repositorio ya está en {REPO_DIR}; se actualiza.")
    resultado = subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
else:
    resultado = subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            f"https://{GITHUB_TOKEN + '@' if GITHUB_TOKEN else ''}github.com/ajgomez23/tfm-compresion-yolov8.git",
            str(REPO_DIR),
        ],
        capture_output=True,
        text=True,
    )

if resultado.returncode != 0:
    raise RuntimeError(
        "Git ha fallado (mensaje censurado por si contenía el token):\n"
        + _censurar(resultado.stderr)[:800]
    )

# Se comprueba que el clon trae lo que hace falta, no solo que git dijera que si.
imprescindibles = [
    "pyproject.toml",
    "requirements.txt",
    "scripts/00_setup_check.py",
    "scripts/04_train_baselines.py",
    "src/tfm_edge",
]
faltan = [f for f in imprescindibles if not (REPO_DIR / f).exists()]
if faltan:
    raise RuntimeError(f"El clon está incompleto: faltan {faltan}")

revision = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h %s"],
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Repositorio listo en {REPO_DIR}")
print(f"  revisión: {revision}")

Salida esperada: la revisión (hash y asunto del último commit), que identifica la versión del código
usada.

## 3. Intérprete y dependencias

El Python de Colab es 3.13 y `numpy==1.26.4` solo publica ruedas hasta `cp312`; la primera versión
con `cp313` es la 2.1.0, incompatible con el `numpy<2.0.0` que exige el extra `export` de
Ultralytics. Por eso se fija el intérprete y no se cambian los pines.

`uv` (fijado a `uv==0.12.5`) solo aporta un CPython 3.12. Es el único pin fuera de
`requirements.txt`, porque crea el venv donde se instala ese fichero. El venv se crea con `--seed`,
que incluye `pip`, y después se instala `requirements.txt` y el paquete con `pip install .`, no en
modo editable.

| Dónde | Qué corre | Por qué |
|---|---|---|
| Kernel de Colab (3.13) | Drive, secretos, `tarfile`, `shutil`, `hashlib` | `google.colab` solo existe en el kernel |
| venv (3.12) | `tfm_edge`, `torch`, `ultralytics` y los `scripts/` | es donde están instalados y donde se entrena |

La celda siguiente define tres funciones: `_correr` para comandos, `en_venv` para fragmentos de
Python y `json_en_venv` para fragmentos que devuelven datos. Los valores se pasan por `sys.argv`, no
interpolados en el fragmento.

Las tres usan un entorno propio en lugar del del kernel. El del kernel incluye
`MPLBACKEND=module://matplotlib_inline.backend_inline`, que hace fallar matplotlib en un proceso sin
pantalla cuando Ultralytics comprueba la fuente Arial; se sustituye por `Agg` y se quita
`PYTHONPATH`. Las variables `COLAB_*` se conservan porque la sección 7 las usa para reconocer Colab.

No hay `%cd`: cada subproceso indica su directorio de trabajo. Un `!comando` suelto no se ejecuta en
el repositorio.

In [ ]:
# Python 3.13 del kernel no tiene ruedas para los pines de requirements.txt, así que se
# crea un venv con CPython 3.12 mediante uv (--seed incluye pip) y se instala con pip.
# Cada subproceso indica su directorio de trabajo.
import json
import os
import subprocess
import sys
from pathlib import Path

VENV_DIR = Path("/content/venv312")
VENV_PY = VENV_DIR / "bin" / "python"

#: Separador con el que los fragmentos devuelven datos al kernel; tiene que coincidir con el
#: que aparece en los CODIGO_* de las celdas.
MARCA_JSON = "##JSON##"

#: Entorno de los subprocesos del venv, derivado del del kernel con dos cambios.
#:
#: MPLBACKEND apunta en Colab a 'module://matplotlib_inline.backend_inline'. Ultralytics
#: importa matplotlib al comprobar la fuente (check_det_dataset llama a check_font), que
#: fija el backend desde la variable al importarse, y en el venv ese backend no existe.
#:
#: PYTHONPATH y similares apuntan a módulos del 3.13 del kernel y tendrían prioridad
#: sobre el site-packages del venv.
#:
#: Las COLAB_* se conservan porque detect_execution_context() las usa para reconocer
#: Colab desde el venv.
ENTORNO_VENV = dict(os.environ)
_RETIRADAS = {
    clave: ENTORNO_VENV.pop(clave)
    for clave in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP")
    if clave in ENTORNO_VENV
}
_MPL_PREVIO = ENTORNO_VENV.get("MPLBACKEND", "")
# Agg es el backend para un proceso sin pantalla; Ultralytics guarda las figuras en
# fichero. Se fija explícitamente para no depender de la detección automática.
ENTORNO_VENV["MPLBACKEND"] = "Agg"


def _correr(comando, cwd=None):
    """Ejecuta un comando y devuelve su salida; lanza RuntimeError si termina con error."""
    proceso = subprocess.run(
        [str(c) for c in comando],
        capture_output=True,
        text=True,
        cwd=str(cwd) if cwd else None,
        env=ENTORNO_VENV,
    )
    if proceso.returncode != 0:
        raise RuntimeError(
            f"FALLO ({proceso.returncode}): {' '.join(str(c) for c in comando)}\n"
            f"{proceso.stdout[-1500:]}\n{proceso.stderr[-1500:]}"
        )
    return proceso.stdout


def en_venv(codigo, *args, cwd=None):
    """Ejecuta un fragmento de Python en el venv; los valores llegan por ``sys.argv[1:]``.

    Devuelve la salida estándar y lanza RuntimeError si el fragmento termina con error.
    """
    proceso = subprocess.run(
        [str(VENV_PY), "-c", codigo, *[str(a) for a in args]],
        capture_output=True,
        text=True,
        cwd=str(cwd or REPO_DIR),
        env=ENTORNO_VENV,
    )
    print(proceso.stdout.split(MARCA_JSON)[0], end="")
    if proceso.returncode != 0:
        raise RuntimeError(
            f"El fragmento ha fallado bajo el venv (código {proceso.returncode}):\n"
            + (proceso.stderr[-2000:].strip() or "sin detalle en stderr")
        )
    return proceso.stdout


def json_en_venv(codigo, *args, cwd=None):
    """Como ``en_venv``, pero devuelve el JSON que el fragmento imprime tras la marca."""
    salida = en_venv(codigo, *args, cwd=cwd)
    if MARCA_JSON not in salida:
        raise RuntimeError(
            f"El fragmento no ha emitido la marca {MARCA_JSON}; no hay datos que leer."
        )
    return json.loads(salida.rsplit(MARCA_JSON, 1)[1])


print(f"Intérprete del kernel: {sys.version.split()[0]}")
print(f"MPLBACKEND heredado  : {_MPL_PREVIO or '(sin definir)'}, sustituido por Agg")
print(f"Retiradas del entorno: {_RETIRADAS or 'ninguna'}")

#: Versión fija de uv. Se fija aquí y no en requirements.txt porque uv crea el venv donde
#: se instala ese fichero.
UV_PIN = "0.12.5"

_correr([sys.executable, "-m", "pip", "install", "-q", f"uv=={UV_PIN}"])
UV_VERSION = _correr([sys.executable, "-m", "uv", "--version"]).strip()
print(f"uv                   : {UV_VERSION}")
if UV_PIN not in UV_VERSION.split():
    raise RuntimeError(
        f"Se ancló uv=={UV_PIN} y 'uv --version' responde {UV_VERSION!r}. El uv "
        "que se ejecuta no es el que pip acaba de instalar; comprueba cuál "
        "proporciona el 3.12 antes de seguir."
    )

if VENV_PY.is_file():
    print(f"El venv ya existe en {VENV_DIR}; no se vuelve a crear.")
else:
    _correr([sys.executable, "-m", "uv", "venv", VENV_DIR, "--python", "3.12", "--seed"])

VENV_VERSION = _correr([VENV_PY, "-c", "import platform; print(platform.python_version())"]).strip()
print(f"Intérprete del venv  : {VENV_VERSION}   ({VENV_PY})")
if not VENV_VERSION.startswith("3.12."):
    raise RuntimeError(
        f"El venv ha salido con Python {VENV_VERSION} y se pedía 3.12. Sin 3.12 los "
        "pines de requirements.txt no tienen rueda."
    )

# Instalación dentro del venv, no editable: el modo -e depende de un .pth que en Colab
# no se procesaba.
print("\nInstalando requirements.txt en el venv (unos minutos)...")
_correr([VENV_PY, "-m", "pip", "install", "-q", "-r", REPO_DIR / "requirements.txt"])
_correr([VENV_PY, "-m", "pip", "install", "-q", str(REPO_DIR)])
print("Dependencias y paquete instalados en el venv.")

In [ ]:
import tempfile

# Las comprobaciones se hacen en el venv, que es donde corre la campaña.
CODIGO_VERSIONES = """
import torch, ultralytics

print(f"  torch        {torch.__version__}")
marca = "OK" if ultralytics.__version__ == "8.3.253" else "AVISO: se esperaba 8.3.253"
print(f"  ultralytics  {ultralytics.__version__:12s} {marca}")

# Se ejecuta una operación real, porque is_available() no basta.
if not torch.cuda.is_available():
    raise SystemExit("el torch del venv no ve la GPU")
x = torch.randn(256, 256, device="cuda")
torch.cuda.synchronize()
print(f"  CUDA         {torch.version.cuda}  ({torch.cuda.get_device_name(0)})")
print(f"  operación real en GPU: suma {float((x @ x).sum()):.1f}")
"""

try:
    en_venv(CODIGO_VERSIONES)
except RuntimeError as error:
    raise RuntimeError(
        f"{error}\n\nSi falta la GPU, cámbialo en Entorno de ejecución, Cambiar tipo de "
        "entorno de ejecución, GPU."
    ) from None

# 'import tfm_edge' se comprueba desde un directorio neutro, porque el cwd por defecto
# de en_venv es el repositorio.
try:
    with tempfile.TemporaryDirectory() as neutral:
        ruta = en_venv("import tfm_edge; print(tfm_edge.__file__)", cwd=neutral).strip()
except RuntimeError as error:
    raise RuntimeError(
        f"{error}\n\ntfm_edge no es importable desde el venv. Reinstala con:\n"
        f"    !{VENV_PY} -m pip install --force-reinstall --no-deps {REPO_DIR}"
    ) from None

if str(VENV_DIR) not in ruta:
    raise RuntimeError(
        f"tfm_edge se ha importado desde '{ruta}', fuera del venv."
    )
print("  tfm_edge     importable desde un intérprete limpio, dentro del venv")
print(f"               {ruta}")
print("\nDependencias correctas.")

Salida esperada: las versiones del venv, la operación en GPU y `tfm_edge` importable desde un
intérprete limpio dentro del venv.

La importación se comprueba desde un directorio neutro, porque el `cwd` por defecto de `en_venv` es
el repositorio, y se comprueba que el fichero importado está dentro del venv. CUDA se comprueba con
una multiplicación real, no solo con `is_available()`.

In [ ]:
# Comprueba que los paquetes compilados funcionan con la numpy instalada. En un venv
# nuevo no debería haber desajustes, pero la comprobación es barata. Cada paquete se
# ejercita en un subproceso propio.
CODIGO_ABI = """
from tfm_edge.common.binary_compat import (
    check_numpy_binary_compat,
    format_binary_compat_report,
)

resultados = check_numpy_binary_compat()
print(format_binary_compat_report(resultados))

rotos = [r for r in resultados if r.installed and not r.healthy]
if rotos:
    raise SystemExit(
        "paquetes incompatibles con la numpy del venv: "
        + ", ".join(r.package for r in rotos)
    )
"""

try:
    en_venv(CODIGO_ABI)
except RuntimeError as error:
    raise RuntimeError(
        f"{error}\n\nReiniciar el entorno de ejecución no lo arregla, porque el venv es "
        "un directorio. Bórralo y repite la "
        f"sección 3:\n    !rm -rf {VENV_DIR}"
    ) from None

Salida esperada: las líneas de la tabla con `[ OK ]`.

`[ ABI]` indica un paquete compilado contra otra versión de NumPy y la celda se detiene; hay que
borrar el venv y repetir la sección 3. `[ -- ]` indica un paquete ausente, que ya informa la celda
anterior.

## 4. Descomprimir el dataset

El `.tar.gz` se descomprime en el disco local de la sesión. Leer miles de imágenes desde Drive haría
de la carga de datos el cuello de botella.

In [ ]:
import tarfile
import time

DATA_DIR = REPO_DIR / "data"
DATASET_DIR = DATA_DIR / "resized_640"
DATA_DIR.mkdir(exist_ok=True)

if DATASET_DIR.is_dir() and (DATASET_DIR / "sh17_ultralytics.yaml").is_file():
    print(f"El dataset ya está en {DATASET_DIR}; no se vuelve a descomprimir.")
else:
    inicio = time.perf_counter()
    print(f"Descomprimiendo {DATASET_TAR.name} ({DATASET_TAR.stat().st_size / 2**30:.2f} GiB)...")
    with tarfile.open(DATASET_TAR, "r:gz") as tar:
        tar.extractall(DATA_DIR)
    print(f"  hecho en {time.perf_counter() - inicio:.0f} s")

# Se comprueba el contenido, no solo que exista el directorio.
yaml_dataset = DATASET_DIR / "sh17_ultralytics.yaml"
num_imagenes = len(list((DATASET_DIR / "images").glob("*")))
num_etiquetas = len(list((DATASET_DIR / "labels").glob("*.txt")))

problemas = []
if not yaml_dataset.is_file():
    problemas.append(f"falta {yaml_dataset}")
if num_imagenes != 8099:
    problemas.append(f"hay {num_imagenes} imágenes y se esperaban 8.099")
if num_etiquetas != 8099:
    problemas.append(f"hay {num_etiquetas} etiquetas y se esperaban 8.099")
if problemas:
    raise RuntimeError("El dataset descomprimido no es correcto: " + "; ".join(problemas))

print(f"Dataset verificado: {num_imagenes:,} imágenes, {num_etiquetas:,} etiquetas")
print(f"  YAML: {yaml_dataset}")

In [ ]:
# Un tar.gz generado con una versión antigua puede llevar rutas absolutas.
CODIGO_PORTABILIDAD = """
import sys
from pathlib import Path

from tfm_edge.data.prepare import assert_dataset_portable

assert_dataset_portable(Path(sys.argv[1]))
print("Portabilidad verificada: ninguna ruta absoluta en el YAML ni en las listas")
"""

try:
    en_venv(CODIGO_PORTABILIDAD, DATASET_DIR)
except RuntimeError as error:
    raise RuntimeError(
        f"{error}\n\n"
        "Si el motivo son rutas absolutas, el tar.gz de Drive se generó con una "
        "versión antigua. Regenéralo en local y súbelo de nuevo:\n"
        "    python scripts/02b_resize_dataset.py --target-size 640 --force\n"
        "    tar -C data -czf sh17_resized640.tar.gz resized_640"
    ) from None

In [ ]:
# Ultralytics tiene que resolver las rutas contra esta ubicación. Se comprueba en el
# venv, que es donde está el Ultralytics que entrena.
CODIGO_ULTRALYTICS = """
import sys
from pathlib import Path

from ultralytics.data.utils import check_det_dataset

yaml_dataset, dataset_dir = Path(sys.argv[1]), Path(sys.argv[2])
datos = check_det_dataset(str(yaml_dataset), autodownload=False)
print(f"Ultralytics acepta el dataset: {datos['nc']} clases")
print(f"  train: {datos['train']}")
print(f"  val  : {datos['val']}")
assert datos["nc"] == 17, f"Se esperaban 17 clases y hay {datos['nc']}"
assert Path(datos["val"]).is_file(), "La lista de validación no se resuelve"
assert str(dataset_dir) in str(datos["val"]), (
    f"Las rutas apuntan fuera de esta sesión: {datos['val']}. El artefacto no es portable."
)
print("")
print("Las rutas se resuelven contra esta sesión, no contra la máquina de origen.")
"""

en_venv(CODIGO_ULTRALYTICS, yaml_dataset, DATASET_DIR)

Salida esperada: 8.099 imágenes, 17 clases y las rutas de `train` y `val` dentro de
`/content/.../data/resized_640/`. Si apuntan a otra máquina, hay que regenerar el `tar.gz`; la celda
anterior lo detecta e indica el comando.

## 5. La referencia de evaluación

Las métricas las calcula pycocotools sobre `data/processed/sh17_coco.json`, con la partición de
`data/splits/val.txt`. Ninguno de los dos está en el repositorio (`data/` no se versiona) ni en
`sh17_resized640.tar.gz`, así que sin ellos la campaña entrenaría y fallaría al evaluar.

No se pueden regenerar aquí: `scripts/02_prepare_dataset.py` valida la procedencia y rechaza una
copia redimensionada (exige resoluciones variadas y un ancho máximo de al menos 1920 px). Además, las
áreas del COCO están en coordenadas originales, de las que depende el criterio de tamaño de objeto.
Por eso el COCO se copia desde Drive (1,8 MiB comprimido).

In [ ]:
import hashlib
import json
import tarfile

# Huellas de los ficheros con los que se evaluó el baseline en local, para medir contra
# la misma referencia que el resto del capítulo 5.
REFERENCIA_ESPERADA = {
    "processed/sh17_coco.json": "f67e72991eac18e1b6573a7c790ccb4b1a29b6c2fdc34d1f2d34526818cc99e7",
    "splits/val.txt": "cb1d27579cb1d218b02c7d64125ef149316cd6ae7234243d01c0306a472ff743",
    "splits/train.txt": "0adc1a8a30bf8f0d18803e98d1e3056b45f7c918d872632e2e87e92ddf10db02",
    "splits/test.txt": "a75a91888f71ab588c384929d7ee1e3fd22cc262832be370d0fcc089ab2e25bc",
}

REFERENCE_TAR = DRIVE_BASE / "sh17_reference.tar.gz"

if not REFERENCE_TAR.is_file():
    disponibles = sorted(p.name for p in DRIVE_BASE.glob("*"))
    raise FileNotFoundError(
        f"Falta '{REFERENCE_TAR}'.\n"
        f"Lo que hay en {DRIVE_BASE}: {disponibles}\n\n"
        "Se genera en local, donde está data/raw, y se sube a "
        "Drive una sola vez:\n"
        "    tar -C data -czf sh17_reference.tar.gz processed/sh17_coco.json splits\n"
        "Son 1,8 MiB. No se puede regenerar en Colab: ver el texto de esta sección."
    )

with tarfile.open(REFERENCE_TAR, "r:gz") as tar:
    tar.extractall(DATA_DIR)
print(f"Referencia extraída de {REFERENCE_TAR.name} en {DATA_DIR}")

# Se comprueba cada fichero y no el tar.gz, cuyo hash depende de gzip y de las marcas de
# tiempo.
print()
discrepancias = []
for relativa, esperado in REFERENCIA_ESPERADA.items():
    ruta = DATA_DIR / relativa
    if not ruta.is_file():
        discrepancias.append(f"{relativa}: no se ha extraído")
        continue
    obtenido = hashlib.sha256(ruta.read_bytes()).hexdigest()
    marca = "OK  " if obtenido == esperado else "MAL "
    print(f"  {marca} {relativa:34s} {obtenido[:16]}...")
    if obtenido != esperado:
        discrepancias.append(f"{relativa}: {obtenido[:16]}... != {esperado[:16]}...")

if discrepancias:
    raise RuntimeError(
        "La referencia de Drive no es la del estudio:\n  "
        + "\n  ".join(discrepancias)
        + "\n\nVuelve a generar el tar.gz en local desde el mismo data/processed y "
        "data/splits con los que se evaluó el baseline."
    )

# Recuentos esperados de la referencia y de la partición de validación.
COCO_PATH = DATA_DIR / "processed" / "sh17_coco.json"
SPLITS_DIR = DATA_DIR / "splits"

coco = json.loads(COCO_PATH.read_text(encoding="utf-8"))
num_val = len([x for x in (SPLITS_DIR / "val.txt").read_text().splitlines() if x.strip()])
print()
print(
    f"COCO   : {len(coco['images']):,} imágenes, {len(coco['annotations']):,} anotaciones, "
    f"{len(coco['categories'])} clases"
)
print(f"val.txt: {num_val:,} imágenes")
assert (len(coco["images"]), len(coco["annotations"]), len(coco["categories"])) == (8099, 75994, 17)
assert num_val == 1213, f"val.txt trae {num_val} imágenes y se esperaban 1.213"

# La misma comprobación que hace el script antes de empezar, ejecutada en el venv.
CODIGO_EVALUACION = """
import sys
from pathlib import Path

from tfm_edge.evaluation.runner import assert_evaluation_inputs_available

coco, splits, yaml_dataset = (Path(a) for a in sys.argv[1:4])
comprobadas = assert_evaluation_inputs_available(coco, splits, "val", yaml_dataset)
print("Las tres piezas de la evaluación están:")
for nombre, ruta in comprobadas.items():
    print(f"  {nombre}: {ruta}")
"""

print()
en_venv(CODIGO_EVALUACION, COCO_PATH, SPLITS_DIR, yaml_dataset)

Salida esperada: las cuatro líneas `OK`, las 8.099 imágenes con 75.994 anotaciones y las 1.213 de
validación. Si alguna huella sale `MAL`, la referencia es distinta de la del baseline y no hay que
continuar.

## 6. Comprobación del entorno

`scripts/00_setup_check.py` comprueba versiones, dependencias, rutas y GPU, y se ejecuta con el
intérprete del venv. La línea `Python` tiene que indicar `3.12.x`; `3.13` significaría que se ha
comprobado el kernel.

In [ ]:
# Con el intérprete del venv; la línea "Python" tiene que indicar 3.12.x.
proceso = subprocess.run(
    [str(VENV_PY), "scripts/00_setup_check.py"],
    capture_output=True,
    text=True,
    cwd=str(REPO_DIR),
    env=ENTORNO_VENV,
)
print(proceso.stdout)
if proceso.returncode != 0:
    print(proceso.stderr[-2000:])
    raise RuntimeError(
        f"00_setup_check.py ha terminado con código {proceso.returncode}. Mira qué "
        "componente obligatorio falta antes de seguir."
    )

Salida esperada: el informe completo del entorno en el que se obtienen los resultados.

## 7. Máquina asignada

Colab asigna máquinas distintas en cada sesión y eso cambia los tiempos. La huella la toma el venv,
que es el que entrena; el kernel añade su versión de Python y la de `uv` y guarda el fichero junto a
los checkpoints, en Drive.

`detect_execution_context()` reconoce Colab por las variables `COLAB_*`, que el subproceso hereda,
porque `import google.colab` no funciona en el venv. Si el contexto no sale `google_colab`, la celda
se detiene: las filas de la sesión quedarían con otra etiqueta. Se corrige en `environment.py`, no
exportando `COLAB_GPU` a mano.

In [ ]:
import json
import os

# La huella la toma el venv, que es el que entrena; el kernel añade su versión de
# Python y la de uv y escribe el fichero en Drive.
CODIGO_HUELLA = """
import json
import os
import platform
import subprocess
from pathlib import Path

import torch

from tfm_edge.common.environment import detect_execution_context


def _cores_fisicos() -> int:
    \"\"\"Cuenta núcleos físicos leyendo /proc/cpuinfo.\"\"\"
    try:
        texto = Path("/proc/cpuinfo").read_text()
        ids = {
            linea.split(":")[1].strip()
            for linea in texto.splitlines()
            if linea.startswith("core id")
        }
        return len(ids) or os.cpu_count() or 0
    except Exception:
        return os.cpu_count() or 0


gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True,
).stdout.strip()

huella = {
    "gpu_nvidia_smi": gpu,
    "gpu_torch": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
    "gpu_memory_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)
    if torch.cuda.is_available()
    else "",
    "cpu_logicos": os.cpu_count(),
    "cpu_fisicos": _cores_fisicos(),
    "ram_gib": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "python_venv": platform.python_version(),
    "execution_context": detect_execution_context(),
}
print("##JSON##")
print(json.dumps(huella))
"""

huella = json_en_venv(CODIGO_HUELLA)
huella["python_kernel"] = sys.version.split()[0]
huella["uv"] = UV_VERSION

for clave, valor in huella.items():
    print(f"  {clave:18s} {valor}")

# El venv no puede importar google.colab, así que detect_execution_context() depende
# de las variables COLAB_*, que el subproceso hereda.
hay_variable = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
print(f"\n  variables COLAB_* en el entorno del kernel: {hay_variable}")
if "google_colab" not in huella["execution_context"]:
    raise RuntimeError(
        "detect_execution_context() no reconoce Colab desde el venv: las filas de "
        f"esta sesión quedarían con contexto '{huella['execution_context']}' en vez "
        "de 'google_colab+container', y no serían comparables con las anteriores.\n"
        "Es un fallo de detección y se corrige en environment.py, no exportando "
        "COLAB_GPU a mano."
    )

destino_huella = RUNS_DIR / "session_environment.json"
historico = []
if destino_huella.is_file():
    historico = json.loads(destino_huella.read_text())
historico.append(huella)
destino_huella.write_text(json.dumps(historico, indent=2, ensure_ascii=False))
print(f"\nHuella añadida a {destino_huella} (sesión {len(historico)})")

Salida esperada: GPU, núcleos físicos y RAM. El baseline se entrenó en una Tesla T4 con 4 núcleos
físicos a 116,5 s por época de extremo a extremo; con otra máquina cambia el presupuesto de la
sección 10.

## 8. Pesos de partida

Antes de lanzar la campaña se comprueba el checkpoint: SHA-256, número de parámetros, número de
clases y que los pesos están en FP32. Un checkpoint en FP16 daría una campaña sin errores pero no
comparable con el resto del estudio.

La copia desde Drive y la SHA-256 se hacen en el kernel. La precisión, los parámetros y las clases
necesitan `Detector`, así que se comprueban en el venv.

In [ ]:
import hashlib
import shutil

# Punto de partida de la campaña.
PUNTO_DE_PARTIDA = "s_pruning_l1global_r25_4e91ddbc"

# Valores medidos en local (torch 2.8.0, ultralytics 8.3.253) sobre estos ficheros y
# qat_default.yaml, sin entrenar. Los identificadores incluyen la huella del checkpoint,
# así que si coinciden aquí el fichero y la configuración son los mismos. La referencia
# FP32 es la fila de pycocotools del registro.
PARTIDAS = {
    "s_baseline_default_745e3d08": {
        "origen_drive": RUNS_DIR / "s_baseline_default" / "weights" / "best.pt",
        "checkpoint": REPO_DIR / "models" / "baselines" / "s_baseline_default" / "best.pt",
        "sha256": "9701d6e68f65d9eb4cef99500101e8da9db6ff227232a5b471523572aec2364a",
        "bytes": 44_868_896,
        "parametros": 11_142_179,
        "qat_id_preparacion": "s_qat_int8-noexcl_45dc3d3b",
        "qat_id_afinado": "s_qat_int8-noexcl_d659d4c2",
        "observadores_peso": 64,
        "observadores_activacion": 121,
        "referencia_fp32": "0.376214",
    },
    "s_pruning_l1global_r50_9419b26f": {
        "origen_drive": DRIVE_BASE
        / "pruned"
        / "s_pruning_l1global_r50_9419b26f"
        / "finetune"
        / "train"
        / "weights"
        / "best.pt",
        "checkpoint": REPO_DIR
        / "models"
        / "pruned"
        / "s_pruning_l1global_r50_9419b26f"
        / "best.pt",
        "sha256": "398bc66ebe2b23405bd812ee3b1318742b7b3b40fa62ee263b158cb1a7e3d349",
        "bytes": 14_767_738,
        "parametros": 3_618_872,
        "qat_id_preparacion": "s_qat_int8-noexcl_039295f3",
        "qat_id_afinado": "s_qat_int8-noexcl_43300d34",
        "observadores_peso": 72,
        "observadores_activacion": 137,
        "referencia_fp32": "0.316322",
    },
    "s_pruning_l1global_r25_4e91ddbc": {
        "origen_drive": DRIVE_BASE
        / "pruned"
        / "s_pruning_l1global_r25_4e91ddbc"
        / "finetune"
        / "train"
        / "weights"
        / "best.pt",
        "checkpoint": REPO_DIR
        / "models"
        / "pruned"
        / "s_pruning_l1global_r25_4e91ddbc"
        / "best.pt",
        "sha256": "c89984a3f5943718d88c6f998b8242838f045c45cc563a21f9e2deebec319cae",
        "bytes": 27_435_578,
        "parametros": 6_781_449,
        "qat_id_preparacion": "s_qat_int8-noexcl_3a795a86",
        "qat_id_afinado": "s_qat_int8-noexcl_4d491880",
        "observadores_peso": 72,
        "observadores_activacion": 137,
        "referencia_fp32": "0.357927",
    },
}
CLASES_ESPERADAS = 17

if PUNTO_DE_PARTIDA not in PARTIDAS:
    raise ValueError(f"PUNTO_DE_PARTIDA '{PUNTO_DE_PARTIDA}' no es ninguno de {sorted(PARTIDAS)}")
PARTIDA = PARTIDAS[PUNTO_DE_PARTIDA]
PARENT_EXPERIMENT_ID = PUNTO_DE_PARTIDA

CHECKPOINT = PARTIDA["checkpoint"]
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
origen_pesos = PARTIDA["origen_drive"]
if not CHECKPOINT.is_file():
    if not origen_pesos.is_file():
        raise FileNotFoundError(
            f"Falta '{origen_pesos}'.\n"
            f"Lo que hay en {origen_pesos.parent}: "
            f"{sorted(p.name for p in origen_pesos.parent.glob('*')) if origen_pesos.parent.is_dir() else 'nada'}"
        )
    shutil.copy2(origen_pesos, CHECKPOINT)
    print(f"Copiado {origen_pesos} en {CHECKPOINT}")

# Ficheros que acompañan al checkpoint, si están en Drive: el coste de entrenamiento
# (para el presupuesto) y la configuración con la que se entrenó.
directorio_ejecucion = origen_pesos.parent.parent
for nombre in ("training_cost.json", "args.yaml", "complexity.json", "schedule.json"):
    fuente = directorio_ejecucion / nombre
    if fuente.is_file() and not (CHECKPOINT.parent / nombre).is_file():
        shutil.copy2(fuente, CHECKPOINT.parent / nombre)

print(f"\nPunto de partida: {PUNTO_DE_PARTIDA}")
print(f"Checkpoint: {CHECKPOINT}  ({CHECKPOINT.stat().st_size / 2**20:.1f} MiB)")
print(
    f"Acompañado de: {sorted(p.name for p in CHECKPOINT.parent.glob('*') if p.name != 'best.pt')}"
)

# La huella se comprueba primero.
huella = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
tamano = CHECKPOINT.stat().st_size
print(f"\nSHA-256    : {huella}")
print(f"             (se esperaba {PARTIDA['sha256']})")
print(f"Bytes      : {tamano:,}   (se esperaban {PARTIDA['bytes']:,})")
if (huella, tamano) != (PARTIDA["sha256"], PARTIDA["bytes"]):
    raise RuntimeError(
        f"El checkpoint no es el medido en local: la huella o el tamaño no coinciden, "
        f"y los identificadores esperados de las secciones 9 y 11 dependen de él."
    )

# Precisión, parámetros y clases se comprueban con Detector en el venv; los valores
# esperados se pasan por sys.argv.
CODIGO_CHECKPOINT = """
import sys
from pathlib import Path

from tfm_edge.models.detector import Detector
from tfm_edge.training.trainer import verify_checkpoint_is_fp32

checkpoint = Path(sys.argv[1])
params_esperados = int(sys.argv[2])
clases_esperadas = int(sys.argv[3])
partida = sys.argv[4]

print(f"Precisión verificada: {verify_checkpoint_is_fp32(checkpoint)}")

detector = Detector.load(checkpoint, input_resolution=640)
parametros = detector.count_parameters()
clases = detector.num_classes
print(f"Parámetros : {parametros:,}   (se esperaban {params_esperados:,})")
print(f"Clases     : {clases}   (se esperaban {clases_esperadas})")

problemas = []
if parametros != params_esperados:
    problemas.append(f"parámetros {parametros:,} != {params_esperados:,}")
if clases != clases_esperadas:
    problemas.append(f"clases {clases} != {clases_esperadas}")
if problemas:
    raise SystemExit(f"El checkpoint no es '{partida}': " + "; ".join(problemas))
"""

print()
en_venv(CODIGO_CHECKPOINT, CHECKPOINT, PARTIDA["parametros"], CLASES_ESPERADAS, PUNTO_DE_PARTIDA)
print(f"\nEs '{PUNTO_DE_PARTIDA}', el mismo fichero que en local.")

Salida esperada: el punto de partida, la SHA-256, los parámetros, las clases y la precisión
verificada, coincidentes con lo esperado. Si no, la celda se detiene.

## 9. Comprobación de la preparación

Prepara el modelo para QAT sin afinar y sin escribir en el registro, y compara el identificador del
experimento y el número de observadores con lo medido en local. El identificador incluye el hash de
la configuración efectiva (también el calentamiento desactivado), así que si coincide, lo que se
afina aquí es lo mismo que se diseñó en local. Si alguna cifra no coincide, no hay que continuar.

In [ ]:
import re
import subprocess
import sys

# Valores medidos en local (torch 2.8.0, ultralytics 8.3.253), tomados de PARTIDA.
QAT_CONFIG = "configs/compression/quantization/qat_default.yaml"
QAT_EXPERIMENT_ID = PARTIDA["qat_id_preparacion"]
QAT_OBSERVADORES_PESO = PARTIDA["observadores_peso"]
QAT_OBSERVADORES_ACTIVACION = PARTIDA["observadores_activacion"]

# VENV_PY y no sys.executable, que es el Python del kernel.
comando = [
    str(VENV_PY),
    "scripts/16_qat.py",
    "--checkpoint",
    str(CHECKPOINT),
    "--config",
    QAT_CONFIG,
    "--parent-experiment-id",
    PARENT_EXPERIMENT_ID,
    # Prepara y comprueba sin afinar.
    "--override",
    "quantization.qat_epochs=0",
    "--no-evaluate",
    "--dry-run",
]
proceso = subprocess.run(comando, cwd=REPO_DIR, capture_output=True, text=True, env=ENTORNO_VENV)
salida = proceso.stdout + proceso.stderr
print(proceso.stdout[-2500:])
if proceso.returncode != 0:
    print(proceso.stderr[-2500:], file=sys.stderr)
    raise RuntimeError(f"La preparación ha fallado con código {proceso.returncode}")

# 1. El identificador del experimento, que lleva el hash de la configuración.
encontrado = re.search(r"Experimento\s*:\s*(\S+)", salida)
if not encontrado:
    raise RuntimeError("No se ha podido leer el identificador del experimento de la salida")
if encontrado.group(1) != QAT_EXPERIMENT_ID:
    raise RuntimeError(
        f"El experimento sale como '{encontrado.group(1)}' y en local salía "
        f"'{QAT_EXPERIMENT_ID}': la configuración efectiva no es la misma."
    )

# 2. Los observadores, que pueden variar con otra versión de PyTorch.
encontrado = re.search(r"observadores\s*:\s*(\d+) en pesos, (\d+) en activaciones", salida)
if not encontrado:
    raise RuntimeError("No se ha podido leer el recuento de observadores de la salida")
peso, activacion = int(encontrado.group(1)), int(encontrado.group(2))
if (peso, activacion) != (QAT_OBSERVADORES_PESO, QAT_OBSERVADORES_ACTIVACION):
    raise RuntimeError(
        f"Observadores: {peso} en pesos y {activacion} en activaciones; en local "
        f"eran {QAT_OBSERVADORES_PESO} y "
        f"{QAT_OBSERVADORES_ACTIVACION}. El afinado no sería comparable."
    )

# 3. El optimizador declarado.
if "SGD, sin calentamiento" not in salida:
    raise RuntimeError(
        "La linea 'Afinado' no declara 'SGD, sin calentamiento'. Revisa la "
        "configuración antes de lanzar el afinado."
    )

# 4. La referencia FP32 que el script lee del registro, con su evaluador entre corchetes.
encontrado = re.search(r"mAP@50-95 = (\S+) \[(\S+), (\S+)\]", salida)
if not encontrado:
    raise RuntimeError(
        "No aparece la línea de la referencia FP32 con su evaluador. Falta la "
        "fila en el registro del clon o el código es de una versión anterior."
    )
referencia, evaluador_ref, dispositivo_ref = encontrado.groups()
if (referencia, evaluador_ref) != (PARTIDA["referencia_fp32"], "pycocotools"):
    raise RuntimeError(
        f"La referencia FP32 sale {referencia} con {evaluador_ref}; en local salía "
        f"{PARTIDA['referencia_fp32']} con pycocotools. El registro del "
        f"clon no es el esperado."
    )

print(f"\nPreparación verificada contra lo medido en local:")
print(f"  experimento  : {QAT_EXPERIMENT_ID}")
print(f"  observadores : {peso} en pesos, {activacion} en activaciones")
print(f"  optimizador  : SGD, sin calentamiento")
print(f"  referencia   : {referencia} [{evaluador_ref}, {dispositivo_ref}]")

Salida esperada: las cuatro líneas del bloque final. Si alguna comprobación falla, la celda se
detiene con el motivo.

La referencia (`0.357927 [pycocotools, colab_t4_gpu]` para `l1global_r25`,
`0.316322 [pycocotools, colab_t4_gpu]` para `l1global_r50` y
`0.376214 [pycocotools, workstation_cpu]` para el baseline) se lee del registro, no se vuelve a
evaluar.

## 10. Presupuesto

El baseline hizo 100 épocas en 11.652,85 s en una Tesla T4 (`training_cost.json`), 116,5 s por época
de extremo a extremo. Las 10 épocas del afinado son el 10 % de ese entrenamiento, lo habitual para un
QAT que parte de un modelo ya convergido.

Otras medidas de referencia con la misma receta (T4, lote 16, FP32 sin AMP): el afinado QAT del
baseline (`s_qat_int8-noexcl_d4d403fa`) tardó 3.265,98 s con un pico de 14.120 MB, y el de
`l1global_r50` (`s_qat_int8-noexcl_43300d34`) 1.868,23 s con 10.442 MB. `l1global_r25` tiene más
parámetros (6.781.449 frente a 3.618.872) y los mismos 72 + 137 observadores; su coste no está
medido.

La cota calculada con el baseline es un mínimo: no incluye el coste de los observadores.

In [ ]:
# La configuración y el coste se leen en el venv con los mismos cargadores que el script.
CODIGO_PRESUPUESTO = """
import json
import sys
from pathlib import Path

from tfm_edge.common.config import load_config
from tfm_edge.training.cost import read_training_cost, seconds_per_epoch

config = load_config(Path(sys.argv[1]))
coste = read_training_cost(Path(sys.argv[2]))
print("##JSON##")
print(json.dumps({
    "qat_epochs": config.quantization.qat_epochs,
    "qat_learning_rate": config.quantization.qat_learning_rate,
    "freeze_observers_epoch": config.quantization.freeze_observers_epoch,
    "batch_size": config.training.batch_size,
    "coste": coste,
    "segundos_epoca": seconds_per_epoch(coste),
}))
"""

PRESUPUESTO = json_en_venv(CODIGO_PRESUPUESTO, REPO_DIR / QAT_CONFIG, CHECKPOINT)
epocas_qat = PRESUPUESTO["qat_epochs"]
tasa_qat = PRESUPUESTO["qat_learning_rate"]

# Afinado QAT completo del baseline: s_qat_int8-noexcl_d4d403fa, T4, lote 16, FP32 sin
# AMP (volcado crudo y fila de experiments).
QAT_BASELINE_SEGUNDOS = 3265.98
QAT_BASELINE_PICO_MB = 14120.0

coste_baseline = PRESUPUESTO["coste"]
segundos_epoca = PRESUPUESTO["segundos_epoca"]

print(f"Afinado pedido: {epocas_qat} epocas, SGD a {tasa_qat:g}, sin calentamiento")
print(f"  observadores congelados en la epoca {PRESUPUESTO['freeze_observers_epoch']}")
print()

if segundos_epoca is None:
    print(
        "AVISO: no hay coste de entrenamiento junto al checkpoint, asi que no se "
        "puede presupuestar con una cifra medida. Se lanza igual; el coste real "
        "lo deja la campaña en su propio volcado."
    )
else:
    print(
        f"Baseline: {coste_baseline['training_epochs']} epocas en "
        f"{coste_baseline['training_time_s']:,.0f} s sobre "
        f"{coste_baseline['training_device']}"
    )
    print(f"  {segundos_epoca:.1f} s por época")
    cota = epocas_qat * segundos_epoca
    print()
    print(
        f"Cota inferior del afinado: {epocas_qat} x {segundos_epoca:.1f} s = "
        f"{cota / 60:.1f} min ({cota:.0f} s)"
    )
    print("  Es un mínimo, no una estimación: no")
    print(
        f"  incluye lo que cuestan los {QAT_OBSERVADORES_PESO + QAT_OBSERVADORES_ACTIVACION} observadores, ni que este afinado"
    )
    print("  corre en FP32 sin AMP mientras que el baseline se midió")
    print("  con AMP. Las dos cosas suman tiempo.")

print()
print(
    f"Afinado QAT del baseline, medido: {QAT_BASELINE_SEGUNDOS:,.0f} s "
    f"({QAT_BASELINE_SEGUNDOS / 60:.1f} min), pico {QAT_BASELINE_PICO_MB:,.0f} MB"
)
print(f"  Punto de partida de esta sesion: {PUNTO_DE_PARTIDA}. Su coste y su memoria")
print("  no están medidos: los mide esta campaña.")

Salida esperada: la cota en minutos y el coste por época del baseline. Con una máquina distinta de
la T4 original, la cota cambia.

## 11. Campaña

Una sola ejecución; el identificador ya se ha comprobado en la sección 9.

In [ ]:
# Configuración de la campaña.
LOTE = 16  # el mismo del afinado QAT del baseline (ver abajo)
WORKERS = 2  # 2 núcleos físicos en la máquina de Colab

SALIDA_QAT = DRIVE_BASE / "qat"
QAT_EXPERIMENT_ID_AFINADO = PARTIDA["qat_id_afinado"]

# El lote tiene que coincidir con training.batch_size del YAML, que es con el que se
# calculó el identificador esperado.
if LOTE != PRESUPUESTO["batch_size"]:
    raise RuntimeError(
        f"LOTE={LOTE} y el YAML declara training.batch_size="
        f"{PRESUPUESTO['batch_size']}. Recalcula el identificador esperado "
        f"antes de lanzar con otro lote."
    )

print(f"Configuración : {QAT_CONFIG}")
print(f"Partida       : {PUNTO_DE_PARTIDA}")
print(f"Experimento   : {QAT_EXPERIMENT_ID_AFINADO}  (esperado, calculado en local)")
print(f"Salida (Drive): {SALIDA_QAT}")
print(f"Lote {LOTE}, {WORKERS} cargadores")

In [ ]:
# Se lanza con subprocess, como en los cuadernos 00 y 01, para comprobar el código de
# salida.
import shutil
import subprocess
import time
from pathlib import Path


def salvar_registro():
    """Copia el registro y el volcado crudo a Drive y devuelve una línea por directorio.

    Se llama siempre: 16_qat.py escribe la fila de experiments antes que la de calidad,
    así que puede haber resultados aunque el script falle.
    """
    # Un subdirectorio por experimento para no sustituir los CSV de otra campaña.
    destino = DRIVE_BASE / "resultados_qat" / QAT_EXPERIMENT_ID_AFINADO
    destino.mkdir(parents=True, exist_ok=True)
    lineas = []
    for subdir in ("registry", "raw/qat"):
        origen = REPO_DIR / "results" / subdir
        if origen.is_dir():
            copia = destino / Path(subdir).name
            shutil.copytree(origen, copia, dirs_exist_ok=True)
            lineas.append(f"{origen} copiado a {copia}: {sorted(p.name for p in copia.glob('*'))}")
        else:
            lineas.append(f"AVISO: no existe {origen}")
    return lineas


# VENV_PY: la campaña corre en el venv de la sección 3.
comando = [
    str(VENV_PY),
    "scripts/16_qat.py",
    "--checkpoint",
    str(CHECKPOINT),
    "--config",
    QAT_CONFIG,
    "--parent-experiment-id",
    PARENT_EXPERIMENT_ID,
    "--batch-size",
    str(LOTE),
    "--workers",
    str(WORKERS),
    # La salida va a Drive para conservar los pesos si la sesión se corta.
    "--output-dir",
    str(SALIDA_QAT),
    # El mAP se mide en la GPU de esta sesión y el device_id lo indica. La fila del
    # capítulo 5 se vuelve a medir en local.
    "--device-id",
    "colab_t4_gpu",
    # 10 épocas con close_mosaic=10, sin mosaico en ningún lote, como el afinado QAT
    # del baseline.
    "--allow-mosaic-free",
]
# El entorno del venv fija MPLBACKEND a 'Agg'; se imprime para que quede en la salida.
print(f"MPLBACKEND que se le pasa: {ENTORNO_VENV.get('MPLBACKEND')!r}")
print(" ".join(comando))
print("-" * 78)

inicio = time.perf_counter()
proceso = subprocess.run(comando, cwd=REPO_DIR, env=ENTORNO_VENV)
transcurrido = time.perf_counter() - inicio

print("-" * 78)
# Se copia a Drive antes de comprobar el resultado, también si ha fallado.
for linea in salvar_registro():
    print(linea)

print("-" * 78)
print(f"Código de salida: {proceso.returncode}   ({transcurrido / 60:.1f} min)")
volcado = REPO_DIR / "results" / "raw" / "qat" / f"{QAT_EXPERIMENT_ID_AFINADO}.json"
print(f"Volcado esperado {volcado.name}: {'PRESENTE' if volcado.is_file() else 'AUSENTE'}")
if proceso.returncode == 0:
    print("  El afinado ha terminado y está inscrito en el registro.")
elif proceso.returncode == 2:
    print("  El afinado ha fallado y el fallo está inscrito con su motivo.")
    print("  Si los observadores no vieron datos, lo entrenado no era QAT,")
    print("  y eso también es un resultado.")
else:
    print("  Se ha detenido por una comprobación previa o por un fallo no controlado;")
    print("  el código de salida no distingue los dos casos.")
    print("  Un fallo después de escribir la fila de experiments también sale")
    print("  con 1. Revisa lo copiado a Drive antes de relanzar la campaña.")
    print("  Si el motivo es falta de memoria de GPU, no relances con otro lote:")
    print("  cambiaría la receta.")
    print("  Antes hay que decidir el lote y recalcular el identificador esperado.")

Salida esperada:

- El tiempo real frente a la cota de la sección 10.
- Las líneas del optimizador que escribe el `Trainer` al arrancar, con el optimizador y la tasa que
  monta Ultralytics.
- Las métricas FP32 del baseline y del afinado, con su diferencia. El INT8 se mide después en cada
  runtime.
- La línea `vieron datos`; si indica `no`, lo entrenado no era QAT y el script lo trata como fallo.
- El código de salida y si el volcado esperado aparece como `PRESENTE`.
- Si falla por memoria de GPU, no se relanza con otro lote.

## 12. Recogida de resultados

El registro y el volcado crudo ya se copian al terminar la sección 11. Esta celda repite la copia y
lista lo que hay:

- El registro, del que salen las tablas.
- El volcado crudo, con la configuración efectiva, las semillas y el informe del afinado.
- Los pesos afinados (`qat_stripped_fp32.pt`), que son la entrada de la materialización INT8 del
  capítulo 5.2, hecha en local por el mismo camino que la rama PTQ.

In [ ]:
# La copia ya la hizo la sección 11; se repite para listar lo que hay.
for linea in salvar_registro():
    print(linea)

# Los pesos ya están en Drive (--output-dir).
if SALIDA_QAT.is_dir():
    print(f"\nPesos en {SALIDA_QAT}:")
    for pesos in sorted(SALIDA_QAT.rglob("*.pt")):
        print(f"  {pesos.relative_to(SALIDA_QAT)}  " f"({pesos.stat().st_size / 2**20:.1f} MiB)")
else:
    print(f"\nAVISO: no existe {SALIDA_QAT}")

Salida esperada: la lista de CSV copiados y la de pesos, que se descargan para incorporarlos al
registro local. Hace falta `qat_stripped_fp32.pt`, el modelo afinado sin los observadores; el
`best.pt` de Ultralytics conserva la cuantización simulada y no es el que se materializa en INT8.